# Notebook 00: Lung cell annotation validation

**Purpose:** Validate the published lung cell labels before defining alveolar macrophages or MHCII-high/low AM states.

**Status:** Local technical-validation notebook. A published `Macrophages` label is treated only as a macrophage candidate pool.

## tl;dr

This notebook first verifies the source object and published labels. It does **not** equate all `Macrophages` with alveolar macrophages and does not assign MHCII states. Executed evidence and limitations are summarized under **Takeaways**.

## Context & Methods

### Cell definitions being checked

- **Macrophage candidate pool:** `celltype_final == "Macrophages"`. This is not yet an AM definition.
- **AT1 candidate:** `celltype_final == "AT1"`.
- **AT2 candidate:** `celltype_final == "AT2"`.

The subsequent marker analysis will compare alveolar-macrophage evidence (`MARCO`, `APOE`) with interstitial-macrophage and monocyte alternatives. No AM inclusion flag will be created in this notebook.

### Key assumptions

The master H5AD is authoritative. `X` is used only after its transformation is reported. `raw.X` is preserved for later count-aware work. Donor is the biological replicate; cells are not independent replicates. Local mode may select cores, but it never subsamples cells inside a selected core for spatial work.

### 1. Parameters

This visible cell selects local or HPC paths. Environment variables can override defaults without editing scientific code.

In [1]:
from pathlib import Path
import os
import sys

# Jupyter normally starts this notebook from notebooks/. Add the repository
# root explicitly so the same reusable module imports locally and on HPC.
WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == 'notebooks' else WORKING_DIR
PIPELINE_PATH = REPO_ROOT / 'scripts' / 'xty_am_pipeline.py'
assert PIPELINE_PATH.exists(), (
    f'Cannot locate {PIPELINE_PATH}. Start Jupyter from the repository root '
    'or its notebooks directory.'
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

RUN_MODE = os.getenv('XTY_AM_RUN_MODE', 'local_test')
RANDOM_SEED = 20260912
LOCAL_DATA_DIR = Path(r'D:\Xiaonan\CODEX_projects\Xiaotong_AM\Spatial\Spatial')
HPC_DATA_DIR = Path('/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Data/External_Data/Xu_NC2026_human/data/Spatial')
DATA_DIR = Path(os.getenv('XTY_AM_DATA_DIR', LOCAL_DATA_DIR if RUN_MODE == 'local_test' else HPC_DATA_DIR))
OUTPUT_ROOT = Path(os.getenv('XTY_AM_OUTPUT_ROOT', Path.cwd() / 'outputs' / RUN_MODE))
DATA_PATH = DATA_DIR / 'xenium.h5ad'
N_LOCAL_CORES = 4
assert RUN_MODE in {'local_test', 'hpc_full'}
print({'run_mode': RUN_MODE, 'data_path': str(DATA_PATH), 'output_root': str(OUTPUT_ROOT)})

{'run_mode': 'local_test', 'data_path': 'D:\\Xiaonan\\CODEX_projects\\Xiaotong_AM\\Spatial\\Spatial\\xenium.h5ad', 'output_root': 'D:\\Xiaonan\\CODEX_projects\\Xiaotong_AM\\XTY_AM_project\\.worktrees\\source-python-script\\outputs\\local_test'}


## Data

### 2. Load the master H5AD in backed mode

Backed mode avoids loading the full expression matrix before genes are requested.

In [2]:
import json
import platform

import anndata as ad
import numpy as np
import pandas as pd

from scripts.xty_am_pipeline import (
    CANONICAL_UNMEASURED_CHECKS,
    CELLTYPE_PALETTE,
    FOCUS_PALETTE,
    MARKER_MODULES,
    compute_program_scores,
    configure_plot_style,
    extract_marker_matrices,
    marker_availability_table,
    plot_focus_umap,
    plot_full_umap,
    plot_marker_dotplot,
    plot_program_umap,
    plot_spatial_celltypes,
    plot_spatial_focus,
    plot_spatial_programs,
    save_figure,
    select_representative_cores,
    summarize_markers,
    validate_xenium_metadata,
)

assert DATA_PATH.exists(), f"Missing input: {DATA_PATH}"
adata = ad.read_h5ad(DATA_PATH, backed="r")
configure_plot_style()
print(f"Loaded {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"Python {platform.python_version()} | anndata {ad.__version__}")
print("Reusable functions: scripts/xty_am_pipeline.py")

Loaded 332,063 cells x 389 genes
Python 3.12.14 | anndata 0.13.3.post0
Reusable functions: scripts/xty_am_pipeline.py


D:\Xiaonan\CODEX_projects\Xiaotong_AM\.analysis_tmp\ipykernel_29216\584979635.py:34: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(f"Python {platform.python_version()} | anndata {ad.__version__}")


### 3. Validate observation grain and spatial coordinates

Each row must be one unique segmented cell. Donor, core, region, published cell type, QC fields, UMAP coordinates, and spatial centroids are required. Coordinates from different cores are never combined as one spatial field.

In [3]:
metadata_audit = validate_xenium_metadata(adata)
display(pd.Series(metadata_audit, name="value").to_frame())

x_probe = adata[:1000, :20].X
raw_probe = adata.raw.X[:1000, :20]
from scipy import sparse
x_probe = x_probe.toarray() if sparse.issparse(x_probe) else np.asarray(x_probe)
raw_probe = raw_probe.toarray() if sparse.issparse(raw_probe) else np.asarray(raw_probe)
assert np.nanmin(x_probe) >= 0 and np.nanmin(raw_probe) >= 0
assert np.allclose(raw_probe, np.round(raw_probe)), "raw.X is not integer-like"
assert np.any(np.abs(x_probe - np.round(x_probe)) > 1e-4), "X does not appear transformed"

expression_semantics = {
    "X": "non-negative transformed/log-normalized expression",
    "raw.X": "non-negative integer-like transcript counts",
    "dotplot_color": "equal-weight donor mean of within-donor mean X",
    "dotplot_size": "equal-weight donor mean of within-donor fraction raw.X > 0",
}
display(pd.Series(expression_semantics, name="interpretation").to_frame())

,value
n_cells,332063
n_genes,389
duplicate_cell_ids,0
missing_required_values,"{'donor_id': 0, 'core_id': 0, 'tissue_annotati..."


,interpretation
X,non-negative transformed/log-normalized expres...
raw.X,non-negative integer-like transcript counts
dotplot_color,equal-weight donor mean of within-donor mean X
dotplot_size,equal-weight donor mean of within-donor fracti...


### 4. Count published labels and select complete local-test cores

Counts below describe source labels, not validated biological identities. Local cores are selected deterministically from eligible alveolar cores across the distribution of core sizes; every cell in each selected core is retained.

In [4]:
obs = adata.obs.copy()
obs["celltype_final"] = obs["celltype_final"].astype(str)
celltype_counts = obs["celltype_final"].value_counts().sort_values(ascending=False)
assert {"Macrophages", "AT1", "AT2"}.issubset(celltype_counts.index)

selected_cores, core_summary = select_representative_cores(
    obs,
    n_cores=N_LOCAL_CORES,
    tissue_code="A",
    required_celltypes=("Macrophages", "AT1", "AT2"),
)
selected_mask = obs["core_id"].astype(str).isin(selected_cores)
selected_obs = obs.loc[selected_mask].copy()

print("Published full-object cell-type counts:")
display(celltype_counts.rename("n_cells").to_frame())
print("Complete alveolar cores used for spatial display:")
display(core_summary.loc[selected_cores])
print(f"Local expression-analysis cells: {selected_mask.sum():,}")

Published full-object cell-type counts:


,n_cells
celltype_final,
Endothelial,74225
Fibroblasts,66767
Monocytes,33479
Macrophages,33375
AT2,21495
AT1,20977
SmoothMuscle,12644
Pericytes,12553
CD8T,10603


Complete alveolar cores used for spatial display:


,donor_id,tissue_annotation,n_cells,Macrophages,AT1,AT2
core_id,,,,,,
D19.c2,D19,A,517,26,73,26
D17.c1,D17,A,3763,154,355,1181
D21.c1,D21,A,5538,545,483,363
D4.c3,D4,A,11865,514,1249,163


Local expression-analysis cells: 21,683


## Results

### 5. Machine-readable audit summary

This bounded JSON is used by the local integration test and records source-label counts without claiming AM identity.

In [5]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary = {
    'run_mode': RUN_MODE,
    'data_path': str(DATA_PATH),
    'n_cells': int(adata.n_obs),
    'n_genes': int(adata.n_vars),
    'duplicate_cell_ids': int(adata.n_obs - adata.obs_names.nunique()),
    'celltype_counts': {str(k): int(v) for k, v in celltype_counts.items()},
    'selected_cores': [str(x) for x in selected_cores],
    'selected_core_cells': int(selected_mask.sum()),
    'am_definition_status': 'not_finalized_macrophage_candidate_pool_only',
}
summary_path = OUTPUT_ROOT / 'notebook_00_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Wrote {summary_path}')
display(pd.Series(summary).to_frame('value'))

Wrote D:\Xiaonan\CODEX_projects\Xiaotong_AM\XTY_AM_project\.worktrees\source-python-script\outputs\local_test\notebook_00_summary.json


,value
run_mode,local_test
data_path,D:\Xiaonan\CODEX_projects\Xiaotong_AM\Spatial\...
n_cells,332063
n_genes,389
duplicate_cell_ids,0
celltype_counts,"{'Endothelial': 74225, 'Fibroblasts': 66767, '..."
selected_cores,"[D19.c2, D17.c1, D21.c1, D4.c3]"
selected_core_cells,21683
am_definition_status,not_finalized_macrophage_candidate_pool_only


### 6. Reusable marker statistics and visual outputs

All reusable scientific functions, marker definitions, and palettes are imported from scripts/xty_am_pipeline.py. This notebook remains the step-wise execution and review layer: it shows parameters, calls each function explicitly, displays intermediate tables, and writes the final outputs.

For each donor and source-label group, mean transformed expression and raw-transcript detection are calculated first. The dotplot then averages donors equally. Program scores remain exploratory: each gene is standardized within the broad macrophage candidate pool before measured genes in a program are averaged. No AM threshold, classifier, differential-expression test, or MHCII state is created here.

In [6]:
FIGURE_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DPI = 180 if RUN_MODE == "local_test" else 300

# Step 6a: report measured and unmeasured marker coverage.
availability_table, marker_availability, present_markers = marker_availability_table(
    adata.var_names,
    marker_modules=MARKER_MODULES,
    additional_checks=CANONICAL_UNMEASURED_CHECKS,
)
availability_table.to_csv(TABLE_DIR / "marker_availability.csv", index=False)
display(availability_table)

# Step 6b: use four intact cores locally; use all cells for HPC summaries.
analysis_idx = (
    np.flatnonzero(selected_mask.to_numpy())
    if RUN_MODE == "local_test"
    else np.arange(adata.n_obs)
)
analysis_obs = obs.iloc[analysis_idx].copy()
xmat, rmat, zero_pattern_concordance = extract_marker_matrices(
    adata,
    row_indices=analysis_idx,
    genes=present_markers,
)

# Step 6c: calculate core, donor, and equal-weight donor marker summaries.
summary_groups = ("AT1", "AT2", "Macrophages", "Monocytes")
marker_summary, marker_by_core, marker_by_donor = summarize_markers(
    xmat,
    rmat,
    analysis_obs,
    genes=present_markers,
    marker_modules=MARKER_MODULES,
    groups=summary_groups,
)
marker_summary.to_csv(TABLE_DIR / "annotation_marker_summary.csv", index=False)
marker_by_core.to_csv(TABLE_DIR / "annotation_marker_summary_by_core.csv", index=False)
marker_by_donor.to_csv(TABLE_DIR / "annotation_marker_summary_by_donor.csv", index=False)
display(marker_summary.head(12))

# Step 6d: plot every deposited UMAP cell with the shared palette.
labels = obs["celltype_final"].astype(str).to_numpy()
umap = np.asarray(adata.obsm["X_umap"])
spatial = np.asarray(adata.obsm["spatial"])

figure = plot_full_umap(umap, labels, palette=CELLTYPE_PALETTE)
display(figure)
save_figure(figure, FIGURE_DIR, "umap_all_celltypes", dpi=FIGURE_DPI)

figure = plot_focus_umap(
    umap,
    labels,
    focus_labels=("Macrophages", "AT1", "AT2"),
    focus_palette=FOCUS_PALETTE,
)
display(figure)
save_figure(
    figure, FIGURE_DIR, "umap_focus_macrophage_AT1_AT2", dpi=FIGURE_DPI
)

gene_order = [gene for genes in MARKER_MODULES.values() for gene in genes]
figure = plot_marker_dotplot(
    marker_summary,
    gene_order=gene_order,
    group_order=summary_groups,
)
display(figure)
save_figure(figure, FIGURE_DIR, "dotplot_annotation_markers", dpi=FIGURE_DPI)

# Step 6e: compute relative AM, IM, and monocyte evidence programs.
expression = pd.DataFrame(xmat, index=analysis_obs.index, columns=present_markers)
score_modules = {
    "AM evidence": MARKER_MODULES["AM evidence"],
    "IM alternative": MARKER_MODULES["IM alternative"],
    "Monocyte alternative": MARKER_MODULES["Monocyte alternative"],
}
score_cells, score_by_core, score_by_donor = compute_program_scores(
    expression,
    analysis_obs,
    program_modules=score_modules,
    candidate_label="Macrophages",
)
score_cells.to_csv(
    TABLE_DIR / "macrophage_candidate_program_scores_local_or_full.csv"
)
score_by_core.to_csv(
    TABLE_DIR / "macrophage_program_summary_by_core.csv", index=False
)
score_by_donor.to_csv(
    TABLE_DIR / "macrophage_program_summary_by_donor.csv", index=False
)
display(score_by_donor)

candidate_mask = analysis_obs["celltype_final"].eq("Macrophages").to_numpy()
candidate_global_indices = analysis_idx[candidate_mask]
program_scores = score_cells[list(score_modules)]
assert score_cells.index.equals(analysis_obs.index[candidate_mask])

figure = plot_program_umap(
    umap,
    candidate_indices=candidate_global_indices,
    program_scores=program_scores,
)
display(figure)
save_figure(
    figure, FIGURE_DIR, "umap_macrophage_identity_features", dpi=FIGURE_DPI
)

# Step 6f: map all cell types, focus labels, and programs spatially.
figure = plot_spatial_celltypes(
    spatial,
    obs,
    core_ids=selected_cores,
    palette=CELLTYPE_PALETTE,
)
display(figure)
save_figure(figure, FIGURE_DIR, "spatial_all_celltypes", dpi=FIGURE_DPI)

figure = plot_spatial_focus(
    spatial,
    obs,
    core_ids=selected_cores,
    focus_labels=("Macrophages", "AT1", "AT2"),
    focus_palette=FOCUS_PALETTE,
)
display(figure)
save_figure(
    figure, FIGURE_DIR, "spatial_focus_macrophage_AT1_AT2", dpi=FIGURE_DPI
)

figure = plot_spatial_programs(
    spatial,
    obs,
    core_ids=selected_cores,
    candidate_global_indices=candidate_global_indices,
    program_scores=program_scores,
)
display(figure)
save_figure(
    figure, FIGURE_DIR, "spatial_macrophage_identity_features", dpi=FIGURE_DPI
)

,gene,available,module
0,AGER,True,AT1 evidence
1,SCEL,True,AT1 evidence
2,SFTPD,True,AT2 evidence
3,PLA2G4F,True,AT2 evidence
4,MARCO,True,AM evidence
5,APOE,True,AM evidence
6,LYVE1,True,IM alternative
7,CD163,True,IM alternative
8,FCGR3A,True,IM alternative
9,MS4A4A,True,IM alternative


,source_celltype,gene,module,n_donors,n_cells,mean_log_normalized_expression,donor_sd_log_normalized_expression,fraction_detected_raw_gt_0,donor_sd_fraction_detected
0,AT1,AGER,AT1 evidence,4,2160,4.298119,2.403656,0.742460,0.369193
1,AT1,AIF1,Pan-macrophage,4,2160,0.505249,0.300982,0.114608,0.065772
2,AT1,APOE,AM evidence,4,2160,0.335612,0.189425,0.077128,0.046126
3,AT1,CD163,IM alternative,4,2160,0.301635,0.162819,0.068806,0.040288
4,AT1,CD68,Pan-macrophage,4,2160,0.839153,0.366913,0.187494,0.096217
5,AT1,CLEC4E,Monocyte alternative,4,2160,0.272048,0.208405,0.065967,0.052207
6,AT1,FCGR3A,IM alternative,4,2160,0.453592,0.270424,0.099752,0.060463
7,AT1,FCN1,Monocyte alternative,4,2160,0.079981,0.073397,0.019969,0.018384
8,AT1,IL1B,Monocyte alternative,4,2160,0.080937,0.047634,0.017603,0.009140
9,AT1,LYVE1,IM alternative,4,2160,0.349102,0.385624,0.085822,0.099443


<Figure size 1050x800 with 1 Axes>

<Figure size 920x720 with 1 Axes>

<Figure size 1450x470 with 2 Axes>

...

<Figure size 1410x470 with 4 Axes>

<Figure size 1280x1160 with 4 Axes>

<Figure size 1240x1140 with 4 Axes>

<Figure size 1410x1600 with 13 Axes>

(WindowsPath('D:/Xiaonan/CODEX_projects/Xiaotong_AM/XTY_AM_project/.worktrees/source-python-script/outputs/local_test/figures/spatial_macrophage_identity_features.png'),
 WindowsPath('D:/Xiaonan/CODEX_projects/Xiaotong_AM/XTY_AM_project/.worktrees/source-python-script/outputs/local_test/figures/spatial_macrophage_identity_features.pdf'))

### 7. Final machine-readable execution record

The JSON and CSV tables make every plotted definition and statistic auditable. The unresolved AM status is recorded deliberately.

In [7]:
summary.update({
    "expression_semantics": expression_semantics,
    "pipeline_source_script": "scripts/xty_am_pipeline.py",
    "expression_zero_pattern_concordance": zero_pattern_concordance,
    "umap_cells_plotted": int(adata.n_obs),
    "spatial_display_strategy": "four_representative_complete_alveolar_cores",
    "marker_availability": marker_availability,
    "missing_markers": [g for g, available in marker_availability.items() if not available],
    "analysis_cells": int(len(analysis_idx)),
    "figure_stems": [
        "umap_all_celltypes","umap_focus_macrophage_AT1_AT2",
        "dotplot_annotation_markers","umap_macrophage_identity_features",
        "spatial_all_celltypes","spatial_focus_macrophage_AT1_AT2",
        "spatial_macrophage_identity_features",
    ],
    "am_definition_status": "not_finalized_macrophage_candidate_pool_only",
    "mhcii_state_status": "not_assigned_in_notebook_00",
})
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
display(pd.Series({
    "source cells":summary["n_cells"], "panel genes":summary["n_genes"],
    "analysis cells":summary["analysis_cells"], "figures":len(summary["figure_stems"]),
    "AM status":summary["am_definition_status"],
}, name="value").to_frame())

,value
source cells,332063
panel genes,389
analysis cells,21683
figures,7
AM status,not_finalized_macrophage_candidate_pool_only


## Takeaways

- The deposited UMAP and all published labels are shown before AM subtyping.
- AT1 and AT2 are checked using measured marker pairs; an absent panel gene such as SFTPC is not interpreted as negative expression.
- Macrophages remains a broad candidate pool. AM evidence is shown beside IM and monocyte alternatives.
- Program scores are descriptive, panel-constrained, and relative within the candidate pool. They are not a final AM gate.
- Cycling AM are not analyzed and MHCII-high/low states are not assigned in notebook 00.
- The next notebook should define the AM inclusion/exclusion rule only after these diagnostic plots are reviewed. Later inference must use donor, not cell, as the biological replicate.